<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/09-math-in-code/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9: Exercises

Twelve exercises: four Easy, five Medium and three Hard. Each one turns a formula into code, and the checks test it the way the lesson did: against exact answers, numerical approximations and simulations.

Many of the checks are about the details that make maths code go wrong quietly: overflow, rounding, modifying the input, and getting the shape of a gradient right.

**Run the setup cell first.**

Rules:

- NumPy only (and Python's `math` module).
- Avoid loops over data rows where the description asks for vectorised code. Loops over shuffles, resamples or training steps are fine.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_stable_softmax(f):
    z2 = np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]])

    def unchanged():
        z = np.array([3.0, 1.0, 2.0])
        f(z)
        return z.tolist()

    _scenarios("stable_softmax", [
        ("small scores", lambda: f(np.array([0.0, 1.0, 2.0])), np.array([0.09003057, 0.24472847, 0.66524096]), None),
        ("large scores don't overflow", lambda: f(np.array([1000.0, 1001.0, 1002.0])),
         np.array([0.09003057, 0.24472847, 0.66524096]), "subtract the maximum before np.exp"),
        ("very negative scores don't give nan", lambda: f(np.array([-1000.0, -1000.0])), np.array([0.5, 0.5]),
         "exp(-1000) is 0; subtracting the maximum fixes this too"),
        ("2D: each row on its own", lambda: f(z2),
         np.array([[0.09003057, 0.24472847, 0.66524096], [1 / 3, 1 / 3, 1 / 3]]),
         "each row needs its own maximum: z.max(axis=-1, keepdims=True)"),
        ("2D rows sum to 1", lambda: f(np.arange(12.0).reshape(3, 4)).sum(axis=1), np.ones(3), "sum along axis=-1 with keepdims=True"),
        ("input is not modified", unchanged, [3.0, 1.0, 2.0], "z - z.max() makes a new array; z -= ... changes the caller's"),
    ])


def check_log_sum_exp(f):
    rng = np.random.default_rng(0)
    small = rng.normal(size=6)
    _scenarios("log_sum_exp", [
        ("matches the direct formula for small values", lambda: f(small), float(np.log(np.sum(np.exp(small)))), None),
        ("two zeros give log 2", lambda: f(np.array([0.0, 0.0])), math.log(2), None),
        ("large values don't overflow", lambda: f(np.array([1000.0, 1000.0])), 1000 + math.log(2),
         "use c + log(sum(exp(z - c))) with c = max(z)"),
        ("very negative values don't give -inf", lambda: f(np.array([-1000.0, -1000.0])), -1000 + math.log(2),
         "exp(-1000) underflows to 0; the max trick fixes it"),
        ("a single value", lambda: f(np.array([5.0])), 5.0, None),
        ("returns a float", lambda: type(f(np.array([1.0, 2.0]))) is float, True, "wrap the result in float(...)"),
    ])


def check_central_difference(f):
    _scenarios("central_difference", [
        ("derivative of sin at 1 is cos(1)", lambda: abs(f(math.sin, 1.0) - math.cos(1.0)) < 1e-9, True,
         "(f(x + h) - f(x - h)) / (2 * h)"),
        ("x cubed at 2 with h = 0.1 (error is exactly h squared)", lambda: f(lambda x: x ** 3, 2.0, 0.1), 12.01,
         "12.61 would mean a forward difference; check the brackets around (2 * h)"),
        ("uses the default h", lambda: abs(f(lambda x: x ** 3, 2.0) - 12.0) < 1e-8, True, None),
        ("a straight line has a constant slope", lambda: f(lambda x: 3 * x - 7, -4.0), 3.0, None),
    ])


def check_cosine_similarity(f):
    _scenarios("cosine_similarity", [
        ("same direction", lambda: f(np.array([1.0, 2.0, 3.0]), np.array([2.0, 4.0, 6.0])), 1.0, None),
        ("opposite directions", lambda: f(np.array([1.0, -1.0]), np.array([-3.0, 3.0])), -1.0, None),
        ("perpendicular", lambda: f(np.array([1.0, 0.0]), np.array([0.0, 5.0])), 0.0, None),
        ("a known angle (45 degrees)", lambda: f(np.array([1.0, 0.0]), np.array([1.0, 1.0])), 1 / math.sqrt(2), None),
        ("works with lists of ints", lambda: f([3, 4], [4, 3]), 0.96, None),
        ("zero vector raises", lambda: f(np.zeros(3), np.ones(3)), Raises(ValueError),
         "check both norms before dividing"),
        ("different lengths raise", lambda: f(np.ones(3), np.ones(4)), Raises(ValueError), None),
        ("returns a float", lambda: type(f(np.ones(2), np.ones(2))) is float, True, "wrap the result in float(...)"),
    ])


def check_numerical_gradient(f):
    def bowl(p):
        return p[0] ** 2 + 3 * p[1] ** 2

    W = np.array([[0.5, -1.0], [2.0, 0.25]])

    def unchanged():
        x = np.array([0.1, 0.7, -0.3])
        before = x.copy()
        f(lambda v: np.sum(np.sin(v)), x)
        return bool(np.array_equal(x, before))

    _scenarios("numerical_gradient", [
        ("a vector", lambda: f(bowl, np.array([1.0, -2.0])), np.array([2.0, -12.0]), None),
        ("a matrix of weights, same shape back", lambda: f(lambda m: np.sum(m ** 2), W), 2 * W,
         "loop over np.ndindex(x.shape) so any shape works"),
        ("integer input", lambda: f(lambda v: np.sum(v ** 2) / 4, np.array([1, 2])), np.array([0.5, 1.0]),
         "work on a float copy: np.array(x, dtype=float)"),
        ("the input is left exactly as it was", unchanged, True,
         "x[i] += h then x[i] -= h doesn't always restore the exact value; work on a copy and save the original"),
        ("matches a known gradient", lambda: f(lambda v: np.sum(np.sin(v)), np.array([0.0, 1.0])),
         np.array([1.0, math.cos(1.0)]), None),
    ])


def check_gradient_descent(f):
    def grad_sq(x):
        return 2 * x

    def unchanged():
        x0 = np.array([3.0, -1.0])
        f(grad_sq, x0, 0.1, 5)
        return x0.tolist()

    _scenarios("gradient_descent", [
        ("halving each step: x squared, lr 0.25", lambda: f(grad_sq, [3.0], 0.25, 3),
         np.array([[3.0], [1.5], [0.75], [0.375]]), None),
        ("path keeps every step, not copies of the last one", lambda: f(grad_sq, np.array([4.0, -2.0]), 0.25, 2),
         np.array([[4.0, -2.0], [2.0, -1.0], [1.0, -0.5]]),
         "x -= ... changes the same array each time; use x = x - ... and append a copy"),
        ("zero steps returns just the start", lambda: f(grad_sq, [1.0, 2.0], 0.1, 0), np.array([[1.0, 2.0]]), None),
        ("shape is (steps + 1, d)", lambda: f(grad_sq, [1.0, 2.0, 3.0], 0.1, 10).shape, (11, 3), None),
        ("a stretched bowl converges with lr 0.09", lambda: bool(np.allclose(
            f(lambda p: np.array([2 * p[0], 20 * p[1]]), [-4.0, 1.5], 0.09, 300)[-1], 0, atol=1e-6)), True, None),
        ("x0 is not modified", unchanged, [3.0, -1.0], "start from a copy: np.array(x0, dtype=float)"),
    ])


def check_cross_entropy(f):
    probs = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.3, 0.3, 0.4]])
    labels = np.array([0, 1, 2])
    expected = float(-np.mean(np.log([0.7, 0.8, 0.4])))
    _scenarios("cross_entropy", [
        ("perfect predictions give 0", lambda: f(np.eye(3), [0, 1, 2]), 0.0, None),
        ("uniform predictions give log(k)", lambda: f(np.full((4, 3), 1 / 3), [0, 1, 2, 1]), math.log(3), None),
        ("a mixed example", lambda: f(probs, labels), expected,
         "pick probs[np.arange(n), labels], then -mean(log(...))"),
        ("labels as a list", lambda: f(probs, [0, 1, 2]), expected, None),
        ("probability 0 gives a large finite loss", lambda: f(np.array([[1.0, 0.0]]), [1]), -math.log(1e-12),
         "clip with np.clip(picked, eps, 1.0) before the log"),
        ("returns a float", lambda: type(f(probs, labels)) is float, True, "wrap the result in float(...)"),
    ])


def check_bayes_posterior(f):
    _scenarios("bayes_posterior", [
        ("the lesson's example", lambda: f(0.01, 0.95, 0.90), 0.0095 / (0.0095 + 0.099), None),
        ("a common condition", lambda: f(0.5, 0.9, 0.9), 0.9, None),
        ("nobody has it", lambda: f(0.0, 0.95, 0.90), 0.0, None),
        ("no false positives means a positive is certain", lambda: f(0.01, 0.95, 1.0), 1.0, None),
        ("probability above 1 raises", lambda: f(1.2, 0.9, 0.9), Raises(ValueError), "check each input is in [0, 1]"),
        ("negative probability raises", lambda: f(0.1, -0.1, 0.9), Raises(ValueError), None),
        ("no possible positives raises", lambda: f(0.0, 0.9, 1.0), Raises(ValueError),
         "if P(positive) is 0, raise instead of dividing by zero"),
    ])


def check_bootstrap_ci(f):
    data = np.random.default_rng(7).normal(0, 1, 400)

    def width(alpha=0.05, stat=np.mean):
        low, high = f(data, stat, n_boot=3000, alpha=alpha, rng=np.random.default_rng(1))
        return high - low

    def reproducible():
        return f(data, np.median, n_boot=500, rng=np.random.default_rng(3)) == f(data, np.median, n_boot=500, rng=np.random.default_rng(3))

    def contains_mean():
        low, high = f(data, np.mean, n_boot=2000, rng=np.random.default_rng(2))
        return low < data.mean() < high

    _scenarios("bootstrap_ci", [
        ("returns two floats", lambda: [type(v) for v in f(data, np.mean, n_boot=200, rng=np.random.default_rng(0))],
         [float, float], "return float(low), float(high)"),
        ("constant data has a zero-width interval", lambda: f(np.full(30, 4.0), np.mean, n_boot=200, rng=np.random.default_rng(0)),
         (4.0, 4.0), None),
        ("resamples with replacement (the interval isn't zero width)", lambda: width() > 0.05, True,
         "without replacement every resample has the same mean; use rng.integers(0, n, size=n)"),
        ("interval for a mean is about 2 x 1.96 x sd / sqrt(n) wide", lambda: 0.17 < width() < 0.22, True,
         "check the percentiles: 100 * alpha / 2 and 100 * (1 - alpha / 2)"),
        ("contains the sample mean", contains_mean, True, None),
        ("a larger alpha gives a narrower interval", lambda: width(alpha=0.2) < width(alpha=0.05), True, None),
        ("uses the statistic it's given (a median ignores one outlier)",
         lambda: f(np.array([1.0] * 9 + [100.0]), np.median, n_boot=2000, rng=np.random.default_rng(0))[1], 1.0,
         "call statistic(resample), not np.mean"),
        ("same seed, same answer", reproducible, True, "use the rng passed in for all the randomness"),
    ])


def check_fit_linear_gd(f):
    rng = np.random.default_rng(5)
    X = rng.normal(size=(200, 3))
    y = X @ np.array([1.5, -2.0, 0.5]) + 4.0 + rng.normal(scale=0.3, size=200)
    exact = np.linalg.lstsq(np.column_stack([X, np.ones(len(X))]), y, rcond=None)[0]

    def fitted():
        w, b, _ = f(X, y, lr=0.1, epochs=500)
        return np.append(w, b)

    def losses():
        return f(X, y, lr=0.1, epochs=50)[2]

    def unchanged():
        X_copy = X.copy()
        f(X_copy, y, lr=0.1, epochs=5)
        return bool(np.array_equal(X_copy, X))

    x1 = rng.normal(size=(100, 1))
    y1 = 3 * x1[:, 0] - 1
    _scenarios("fit_linear_gd", [
        ("matches the exact least-squares answer", fitted, exact,
         "grad_w = 2 / n * X.T @ error and grad_b = 2 * mean(error), with error = X @ w + b - y"),
        ("one feature, no noise", lambda: np.append(*f(x1, y1, lr=0.1, epochs=800)[:2]), np.array([3.0, -1.0]), None),
        ("losses has one value per epoch", lambda: losses().shape, (50,), None),
        ("the first loss is mean(y squared), before any update", lambda: losses()[0], float(np.mean(y ** 2)),
         "record the loss at the start of the epoch, then update"),
        ("the loss goes down every epoch", lambda: bool(np.all(np.diff(losses()) < 0)), True, None),
        ("b is a float", lambda: type(f(X, y, epochs=3)[1]) is float, True, "return float(b)"),
        ("X is not modified", unchanged, True, None),
    ])


def check_logistic_loss_and_grad(f):
    rng = np.random.default_rng(11)
    X = rng.normal(size=(30, 3))
    y = (rng.random(30) < 0.5).astype(float)
    w = np.array([0.4, -1.2, 0.8])
    b = 0.3

    def numeric():
        h = 1e-6
        gw = np.zeros(3)
        for i in range(3):
            step = np.zeros(3)
            step[i] = h
            gw[i] = (f(w + step, b, X, y)[0] - f(w - step, b, X, y)[0]) / (2 * h)
        gb = (f(w, b + h, X, y)[0] - f(w, b - h, X, y)[0]) / (2 * h)
        return gw, gb

    def grads_match():
        _, gw, gb = f(w, b, X, y)
        nw, nb = numeric()
        return bool(np.allclose(gw, nw, atol=1e-6)) and abs(gb - nb) < 1e-6

    z = X @ w + b
    p = 1 / (1 + np.exp(-z))
    expected_loss = float(np.mean(-y * np.log(p) - (1 - y) * np.log(1 - p)))
    big = np.array([[1000.0]])

    def finite_big():
        loss0, gw0, _ = f(np.array([1.0]), 0.0, big, np.array([0.0]))
        loss1, gw1, _ = f(np.array([1.0]), 0.0, big, np.array([1.0]))
        return (round(loss0, 6), round(float(gw0[0]), 6), round(loss1, 6), round(float(gw1[0]), 6))

    _scenarios("logistic_loss_and_grad", [
        ("loss with all-zero weights is log 2", lambda: f(np.zeros(3), 0.0, X, y)[0], math.log(2), None),
        ("loss matches the formula", lambda: f(w, b, X, y)[0], expected_loss, None),
        ("gradients match a numerical check", grads_match, True,
         "dL/dz = sigmoid(z) - y; grad_w = X.T @ (p - y) / n, grad_b = mean(p - y)"),
        ("shapes and types", lambda: (type(f(w, b, X, y)[0]), f(w, b, X, y)[1].shape, type(f(w, b, X, y)[2])),
         (float, (3,), float), "return float(loss), grad_w (shape (d,)), float(grad_b)"),
        ("huge scores stay finite: (loss y=0, grad y=0, loss y=1, grad y=1)", finite_big, (1000.0, 1000.0, 0.0, 0.0),
         "use np.logaddexp(0, z) - y * z for the loss, and a stable sigmoid"),
    ])


def check_permutation_test(f):
    rng = np.random.default_rng(21)
    same_a, same_b = rng.normal(0, 1, 40), rng.normal(0, 1, 40)
    far_a, far_b = np.arange(100.0, 120.0), np.arange(0.0, 20.0)

    def unchanged():
        a, b = np.array([5.0, 1.0, 3.0]), np.array([2.0, 8.0])
        f(a, b, n_perm=50, rng=np.random.default_rng(0))
        return (a.tolist(), b.tolist())

    def reproducible():
        return f(same_a, same_b, n_perm=300, rng=np.random.default_rng(4)) == f(same_a, same_b, n_perm=300, rng=np.random.default_rng(4))

    _scenarios("permutation_test", [
        ("observed difference is mean(a) - mean(b)", lambda: f(far_a, far_b, n_perm=10, rng=np.random.default_rng(0))[0], 100.0, None),
        ("clearly different groups: p = 1 / (n_perm + 1)", lambda: f(far_a, far_b, n_perm=2000, rng=np.random.default_rng(0))[1],
         1 / 2001, "p = (count + 1) / (n_perm + 1)"),
        ("two-sided: swapping the groups gives the same tiny p", lambda: f(far_b, far_a, n_perm=2000, rng=np.random.default_rng(0))[1],
         1 / 2001, "compare abs(diff) >= abs(observed)"),
        ("small case worked out by hand: p close to 1/3",
         lambda: 0.30 < f([1.0, 2.0], [3.0, 4.0], n_perm=6000, rng=np.random.default_rng(1))[1] < 0.37, True,
         "2 of the 6 possible splits give a difference of size 2"),
        ("groups from the same distribution: p is not small",
         lambda: f(same_a, same_b, n_perm=2000, rng=np.random.default_rng(2))[1] > 0.2, True, None),
        ("inputs are not modified", unchanged, ([5.0, 1.0, 3.0], [2.0, 8.0]),
         "shuffle a pooled copy, e.g. rng.permutation(np.concatenate([a, b]))"),
        ("same seed, same answer", reproducible, True, None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `stable_softmax`

Write `stable_softmax(z)` that returns the softmax of `z` along its **last axis**.

- `z` can be 1D (one set of scores) or 2D (one row of scores per example). Every row of the result sums to 1.
- It must work for very large and very negative scores, such as `[1000, 1001]` or `[-1000, -1000]`, without returning `nan`.
- Don't modify `z`.

Use the trick from the lesson: subtract the maximum before calling `np.exp`. For a 2D input, think about **which** maximum.

In [ ]:
def stable_softmax(z):
    # your code here
    pass

In [ ]:
check_stable_softmax(stable_softmax)

### Exercise 2: `log_sum_exp`

Write `log_sum_exp(z)` that returns `log(sum(exp(z)))` for a 1D array, as a float.

The direct formula overflows for `[1000, 1000]` (the answer is about 1000.69) and underflows for `[-1000, -1000]` (about -999.31). Your version must handle both.

$$\log \sum_j e^{z_j} = c + \log \sum_j e^{z_j - c}, \quad c = \max(z)$$

In [ ]:
def log_sum_exp(z) -> float:
    # your code here
    pass

In [ ]:
check_log_sum_exp(log_sum_exp)

### Exercise 3: `central_difference`

Write `central_difference(f, x, h=1e-5)` that estimates the derivative of a one-variable function `f` at `x` using the central difference:

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

Use the `h` that's passed in.

In [ ]:
def central_difference(f, x: float, h: float = 1e-5) -> float:
    # your code here
    pass

In [ ]:
check_central_difference(central_difference)

### Exercise 4: `cosine_similarity`

Write `cosine_similarity(a, b)` that returns the cosine of the angle between two 1D vectors, as a float:

$$\cos\theta = \frac{a \cdot b}{\|a\|\,\|b\|}$$

Raise `ValueError` if the vectors have different lengths, or if either one is all zeros (its direction is undefined).

In [ ]:
def cosine_similarity(a, b) -> float:
    # your code here
    pass

In [ ]:
check_cosine_similarity(cosine_similarity)

## Medium

### Exercise 5: `numerical_gradient`

Write `numerical_gradient(f, x, h=1e-5)` that estimates the gradient of a function `f` that takes an array and returns a number.

- `x` can be any shape (a vector, or a matrix of weights). The gradient has the same shape as `x`, as floats.
- Use the central difference for each element.
- **Don't modify `x`**, not even temporarily in a way that leaves it slightly different afterwards.
- Integer input like `np.array([1, 2])` must work.

In [ ]:
def numerical_gradient(f, x, h: float = 1e-5):
    # your code here
    pass

In [ ]:
check_numerical_gradient(numerical_gradient)

### Exercise 6: `gradient_descent`

Write `gradient_descent(grad, x0, lr, steps)` that runs gradient descent and returns the **whole path**.

- `grad(x)` returns the gradient at `x` (an array the same shape as `x`).
- Take `steps` steps of `x = x - lr * grad(x)`.
- Return a float array of shape `(steps + 1, d)`: the starting point, then the point after each step.
- `x0` may be a list. Don't modify it.

In [ ]:
def gradient_descent(grad, x0, lr: float, steps: int):
    # your code here
    pass

In [ ]:
check_gradient_descent(gradient_descent)

### Exercise 7: `cross_entropy`

Write `cross_entropy(probs, labels, eps=1e-12)` that returns the **mean** cross-entropy loss as a float.

- `probs` has shape `(n, k)`: each row is a predicted probability distribution over `k` classes.
- `labels` holds the true class index for each row (a list or an integer array).
- The loss for one example is `-log(probability given to its true class)`.
- Clip probabilities to at least `eps`, so a probability of 0 gives a large finite loss instead of infinity.

Use NumPy indexing, not a loop.

In [ ]:
def cross_entropy(probs, labels, eps: float = 1e-12) -> float:
    # your code here
    pass

In [ ]:
check_cross_entropy(cross_entropy)

### Exercise 8: `bayes_posterior`

Write `bayes_posterior(prior, sensitivity, specificity)` that returns the probability of having a condition given a **positive** test result:

- `prior`: the fraction of people with the condition.
- `sensitivity`: P(positive | condition).
- `specificity`: P(negative | no condition).

Raise `ValueError` if any input is outside `[0, 1]`. If nobody can test positive at all, also raise `ValueError` (the question has no answer).

In [ ]:
def bayes_posterior(prior: float, sensitivity: float, specificity: float) -> float:
    # your code here
    pass

In [ ]:
check_bayes_posterior(bayes_posterior)

### Exercise 9: `bootstrap_ci`

Write `bootstrap_ci(data, statistic, n_boot=5000, alpha=0.05, rng=None)` that returns a percentile bootstrap confidence interval `(low, high)` as two floats.

1. Draw `n_boot` resamples of `data`, each the same size as `data`, **with replacement**.
2. Compute `statistic(resample)` for each (e.g. `np.mean` or `np.median`).
3. Return the `alpha/2` and `1 - alpha/2` percentiles of those results.

Use `rng` (a NumPy `Generator`) for the randomness, or `np.random.default_rng()` if it's `None`, so results are reproducible when a seeded generator is passed in.

In [ ]:
def bootstrap_ci(data, statistic, n_boot: int = 5000, alpha: float = 0.05, rng=None):
    # your code here
    pass

In [ ]:
check_bootstrap_ci(bootstrap_ci)

## Hard

### Exercise 10: `fit_linear_gd`

Write `fit_linear_gd(X, y, lr=0.1, epochs=500)` that fits linear regression with full-batch gradient descent on the mean squared error.

- `X` has shape `(n, d)`, `y` has shape `(n,)`. Start from `w = zeros(d)` and `b = 0`.
- The model is `pred = X @ w + b`, and the loss is `mean((pred - y)²)`.
- Each epoch: compute the loss, record it, then update `w` and `b` using the gradients. Work out the gradients yourself.
- Return `(w, b, losses)`: `w` as a float array, `b` as a float, and `losses` as an array of length `epochs`, where `losses[i]` is the loss **before** update `i`.

The checks compare your answer with the exact least-squares solution, so the gradients must be right. They also check `X` isn't modified.

In [ ]:
def fit_linear_gd(X, y, lr: float = 0.1, epochs: int = 500):
    # your code here
    pass

In [ ]:
check_fit_linear_gd(fit_linear_gd)

### Exercise 11: `logistic_loss_and_grad`

Write `logistic_loss_and_grad(w, b, X, y)` for binary logistic regression. Return `(loss, grad_w, grad_b)`:

- `z = X @ w + b`, `p = sigmoid(z) = 1 / (1 + exp(-z))`.
- `loss` is the mean log loss: `mean(-y log p - (1 - y) log(1 - p))`, as a float.
- `grad_w` (shape `(d,)`) and `grad_b` (a float) are its exact gradients. The result simplifies nicely: work it out with the chain rule.

It must stay finite for huge scores. With `z = 1000` and `y = 0`, the loss is 1000, not `inf` or `nan`. Hint: the loss for one example can be rewritten as `log(1 + exp(z)) - y·z`, and `np.logaddexp(0, z)` computes `log(1 + exp(z))` stably.

The checks compare your gradient with a numerical one.

In [ ]:
def logistic_loss_and_grad(w, b: float, X, y):
    # your code here
    pass

In [ ]:
check_logistic_loss_and_grad(logistic_loss_and_grad)

### Exercise 12: `permutation_test`

Write `permutation_test(a, b, n_perm=10_000, rng=None)` for a difference in means. Return `(observed, p_value)`:

- `observed = mean(a) - mean(b)`, as a float.
- Shuffle the pooled values `n_perm` times. Each time, split them into groups the sizes of `a` and `b` and compute the difference in means.
- `p_value = (count + 1) / (n_perm + 1)`, where `count` is the number of shuffles with `|difference| >= |observed|`. That makes it **two-sided** and never exactly 0.
- Use `rng` (or `np.random.default_rng()` if `None`). Don't modify `a` or `b`.

In [ ]:
def permutation_test(a, b, n_perm: int = 10_000, rng=None):
    # your code here
    pass

In [ ]:
check_permutation_test(permutation_test)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where `logistic_loss_and_grad`, the gradient check, the bootstrap and the permutation test come together in a classifier built and tested entirely by hand.